# Impulse and Torque MLP (simple)

This notebook trains a physics-structured and physics informed MLP for a rigid body interacting with a plane-
As input we get the rotation as a 6-D rotation Matrix (with cos and sin)
We parameterise normal force with Hooke, similar to our simulations
We internally predict the contact normal
We couple torque to force via cross product: torque = r_{lever} x f
In the loss we use a Hube loss with a weight for energy conservation

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import json
import math
from tqdm import tqdm

In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

Using device: cuda


## Constants

In [3]:
TRAIN_TEST_SPLIT = 0.8

## Colab Only - Download data

In [4]:
def is_colab():
    try:
        import google.colab
        return True
    except Exception as e:
        return False

if is_colab():

    from google.colab import drive
    from tqdm import tqdm
    import os

    drive.mount('/content/drive')

    src_path = "/content/drive/MyDrive/final_output_contact_points.json"
    dst_path = "/content/final_output_contact_points.json"


    chunk_size = 1024 * 1024  # 1 MB
    file_size = os.path.getsize(src_path)

    with open(src_path, 'rb') as src, open(dst_path, 'wb') as dst:
        with tqdm(total=file_size, unit='B', unit_scale=True, desc="Copying to /content") as pbar:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))

    print("Done! File is now in:", dst_path)
    # Print the first entry
    with open(dst_path, 'r') as f:
        data = json.load(f)

    first = data[0] if isinstance(data, list) else next(iter(data.values()))
    print("\nFirst entry:")
    print(json.dumps(first, indent=2))

Mounted at /content/drive


Copying to /content: 100%|██████████| 2.17G/2.17G [00:28<00:00, 77.3MB/s]


Done! File is now in: /content/final_output_contact_points.json

First entry:
{
  "self_position": {
    "x": -3.4637594043166005e-15,
    "y": 7.782708701896175e-15,
    "z": 0.2868435796504531
  },
  "linear_velocity": {
    "x": -4.2994876397942885e-16,
    "y": 3.5425959677157766e-16,
    "z": 0.013381106343287492
  },
  "angular_velocity": {
    "x": 2.4671324653609616,
    "y": -0.8768681421730111,
    "z": -1.975896532688764e-15
  },
  "self_rotation": {
    "qx": -0.26605885444432054,
    "qy": -0.6108208041608848,
    "qz": 0.6403014686034242,
    "qw": 0.3822625543789557,
    "roll": -1.457426864207349,
    "pitch": -0.12661008256761544,
    "yaw": 2.1782085676680407
  },
  "collider_position": {
    "x": 0.0,
    "y": 0.0,
    "z": 0.0
  },
  "collider_rotation": {
    "qx": 0.0,
    "qy": 0.0,
    "qz": 0.0,
    "qw": 1.0,
    "roll": 0.0,
    "pitch": -0.0,
    "yaw": 0.0
  },
  "relative_position_to_collider": {
    "x": -3.4637594043166005e-15,
    "y": 7.782708701896175

## Dataset

Here we get the contat points with individual forces. From these forces, we calculate the torque and sum up the forces and torques to one force and one torque

In [ ]:
class ContactDataset(Dataset):
    """World-frame wrench dataset for a rigid body on a plane.

    Features (10-D):
        [v_x, v_y, v_z,                           # linear velocity (world frame)
         rel_pos_z,                               # height above plane
         sin(roll), cos(roll),
         sin(pitch), cos(pitch),
         sin(yaw), cos(yaw)]

    Targets (world frame, PHYSICAL UNITS — not normalised):
        force:  (3,)   sum of per-contact forces
        torque: (3,)   sum of (r_world - com_world) x f_world
    """

    def __init__(self, data_list):
        features, forces, torques, collisions = [], [], [], []
        lin_vels, ang_vels = [], []

        for contact in data_list:
            rel_pos = contact["relative_position_to_collider"]
            rel_rot = contact["relative_rotation_to_collider"]
            lin_vel = contact["linear_velocity"]
            ang_vel = contact["angular_velocity"]
            cube_pos = contact["self_position"]

            roll, pitch, yaw = rel_rot["roll"], rel_rot["pitch"], rel_rot["yaw"]
            feats = np.array([
                lin_vel["x"], lin_vel["y"], lin_vel["z"],
                rel_pos["z"],
                np.sin(roll), np.cos(roll),
                np.sin(pitch), np.cos(pitch),
                np.sin(yaw), np.cos(yaw),
            ], dtype=np.float32)


            cube_pos_numpy = np.array([cube_pos["x"], cube_pos["y"], cube_pos["z"]])
            force_numpy = np.zeros(3, dtype=np.float32)
            torque_numpy = np.zeros(3, dtype=np.float32)

            for p in contact.get("points", []):
                p_force = p["force"]
                lever_pos = p["contact_position_world"]
                point_force_numpy = np.array([p_force["x"], p_force["y"], p_force["z"]], dtype=np.float32)
                lever_rel_pos = np.array([lever_pos["x"], lever_pos["y"], lever_pos["z"]], dtype=np.float32) - cube_pos_numpy

                force_numpy += point_force_numpy
                torque_numpy += np.cross(lever_rel_pos, point_force_numpy)

            features.append(feats)
            forces.append(force_numpy)
            torques.append(torque_numpy)
            collisions.append([min(len(contact.get("points", [])), 1)])
            lin_vels.append([lin_vel["x"], lin_vel["y"], lin_vel["z"]])
            ang_vels.append([ang_vel["x"], ang_vel["y"], ang_vel["z"]])

        
        self.features   = torch.FloatTensor(np.asarray(features))
        self.forces     = torch.FloatTensor(np.asarray(forces))
        self.torques    = torch.FloatTensor(np.asarray(torques))
        self.collisions = torch.FloatTensor(np.asarray(collisions))
        
        self.lin_vels   = torch.FloatTensor(np.asarray(lin_vels, dtype=np.float32))
        self.ang_vels   = torch.FloatTensor(np.asarray(ang_vels, dtype=np.float32))


    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (
            self.features[idx],
            {
                "force":        self.forces[idx],
                "torque":       self.torques[idx],
                "is_collision": self.collisions[idx],
                "lin_vel":      self.lin_vels[idx],
                "ang_vel":      self.ang_vels[idx],
            },
        )

### Show dataset

show the first entry of the dataset

In [6]:
json_file = 'final_output_contact_points.json'
with open(json_file, 'r') as f:
    data = json.load(f)
if isinstance(data, dict):
    data = [data]

full_dataset = ContactDataset(data)


Print the first entry of the dataset

In [7]:
print(full_dataset[0])

(tensor([-4.2995e-16,  3.5426e-16,  1.3381e-02,  2.8684e-01, -9.9358e-01,
         1.1313e-01, -1.2627e-01,  9.9200e-01,  8.2113e-01, -5.7074e-01]), {'force': tensor([ 0.0000,  0.0000, 19.5900]), 'torque': tensor([-1.0279, -5.9010,  0.0000]), 'is_collision': tensor([1.]), 'lin_vel': tensor([-4.2995e-16,  3.5426e-16,  1.3381e-02]), 'ang_vel': tensor([ 2.4671e+00, -8.7687e-01, -1.9759e-15])})


Showing some info of the dataset

In [8]:
print("collisions:", int(full_dataset.collisions.sum().item()),
      "/", len(full_dataset))
mask = full_dataset.collisions.squeeze(-1).bool()
if mask.any():
    f_rms = full_dataset.forces[mask].pow(2).mean().sqrt().item()
    t_rms = full_dataset.torques[mask].pow(2).mean().sqrt().item()
    print(f"\nContact-only RMS force  = {f_rms:.4f}")
    print(f"Contact-only RMS torque = {t_rms:.4f}")
    print(f"Suggested w_torque / w_force ratio ≈ {f_rms / max(t_rms, 1e-8):.3f}")


collisions: 969053 / 1669922

Contact-only RMS force  = 3051.5032
Contact-only RMS torque = 373.2209
Suggested w_torque / w_force ratio ≈ 8.176


Split the dataset into train and test dataset and create data loaders

In [9]:
train_size = int(TRAIN_TEST_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator = torch.Generator())

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True,
                          num_workers=min(8, (__import__('os').cpu_count() or 2)))
val_loader   = DataLoader(val_dataset,   batch_size=1024, shuffle=False,
                          num_workers=min(8, (__import__('os').cpu_count() or 2)))


Validation checks on the dataset:

In [ ]:
input_dim = full_dataset.features.shape[1]
print(f"input_dim={input_dim}  (expect 10)")
print(f"force target shape = {tuple(full_dataset.forces.shape)}  (expect (N, 3))")
print(f"torque target shape = {tuple(full_dataset.torques.shape)}  (expect (N, 3))")
assert max(full_dataset.collisions) == 1

## Model

Here is our physically structured model

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

HEAD_OUT_DIM = 11  # 1 (collision) + 1 (depth) + 3 (force_residual) + 3 (normal) + 3 (lever)
VEL_SLICE = slice(0, 3)  # [vx, vy, vz] are the first 3 features


class PostNormResidualMLP(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.fc1(x)))
        out = self.bn2(self.fc2(out))
        out += identity
        return self.act(out)
    
class WrenchPredictor(nn.Module):
    """
    Predicts net wrench (force + torque) with a physics-inspired head.

    Semantics (note: 'per-contact' names but aggregated meaning because targets
    are net wrench across all contact points):
      - depth:          effective penetration (<= 0), scales the spring force
      - contact_normal: direction of net force, unit vector
      - force_residual: 3-vector correction (NOT constrained to the normal)
      - lever:          effective lever arm for torque

    Force assembly:
      F_spring  = -k * depth          (positive scalar; depth <= 0)
      F_damping = -c * (v . n)        (scalar, opposes normal-component of velocity)
      F_mag     = softplus(F_spring + F_damping)   (keeps positivity, gradient-safe)
      F_vec     = F_mag * n + force_residual       (residual can fix direction errors)
      T_vec     = lever x F_vec
    """

    def __init__(self, input_dim=10, width=128, num_blocks=4,
                 baseline_k=1e3, learn_k=False,
                 baseline_bounciness=0.5, learn_bounciness=True,
                 baseline_mass=1.0, learn_mass=False,
                 head_hidden=64):
        super().__init__()

        # --- backbone ---
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.LayerNorm(width),
            nn.GELU(),
        )
        self.backbone = nn.Sequential(
            *[ResidualBlock(width) for _ in range(num_blocks)]
        )

        self.head_trunk = nn.Sequential(
            nn.Linear(width, head_hidden),
            nn.LayerNorm(head_hidden),
            nn.GELU(),
        )
        self.head_out = nn.Linear(head_hidden, HEAD_OUT_DIM)

        # Physics parameters.
        # Defaults: freeze k and mass (you know them from the sim), learn bounciness.
        # Flip learn_k=True only if you want to let the model discover it.
        self.k = nn.Parameter(
            torch.tensor(baseline_k, dtype=torch.float32),
            requires_grad=learn_k,
        )
        self.bounciness = nn.Parameter(
            torch.tensor(baseline_bounciness, dtype=torch.float32),
            requires_grad=learn_bounciness,
        )
        self.mass = nn.Parameter(
            torch.tensor(baseline_mass, dtype=torch.float32),
            requires_grad=learn_mass,
        )
    def forward(self, x):
        h = self.input_proj(x)
        h = self.backbone(h)

        raw = self.head_out(self.head_trunk(h))
        collision_logit, depth_raw, force_residual, normal_raw, lever = raw.split(
            [1, 1, 3, 3, 3], dim=-1
        )

        k_pos    = F.softplus(self.k)     if self.k.requires_grad else self.k
        mass_pos = F.softplus(self.mass)  if self.mass.requires_grad else self.mass
        bounciness = torch.sigmoid(self.bounciness)

        # Critical damping
        c = 2.0 * torch.sqrt(k_pos * mass_pos) * bounciness

        # Prevent negative penetration
        depth = -F.softplus(depth_raw)

        # clamp the normal
        contact_normal = F.normalize(normal_raw, dim=-1, eps=1e-6)

        F_spring = -k_pos * depth

        # Damping force
        velocity = x[:, VEL_SLICE]
        vel_normal = (velocity * contact_normal).sum(dim=-1, keepdim=True)
        F_damping = -c * vel_normal

        F_mag = F.softplus(F_spring + F_damping)

        force_vec  = F_mag * contact_normal + force_residual
        torque_vec = torch.cross(lever, force_vec, dim=-1)

        return {
            "collision_logit": collision_logit,
            "force":  force_vec,
            "torque": torque_vec,
            "aux": {
                "contact_normal": contact_normal,
                "lever":          lever,
                "depth":          depth,
                "k":              k_pos.detach() if torch.is_tensor(k_pos) else k_pos,
                "c":              c.detach(),
                "mass":           mass_pos.detach() if torch.is_tensor(mass_pos) else mass_pos,
                "F_mag":          F_mag,
                "F_spring":       F_spring,
                "F_damping":      F_damping,
                "force_residual": force_residual,
            },
        }


def make_fast_predictor(input_dim=10, width=128, num_blocks=4, baseline_k=1e3):
    model = WrenchPredictor(input_dim=input_dim,
                            width=width, num_blocks=num_blocks,
                            baseline_k=baseline_k)
    try:
        compiled = torch.compile(model, mode="default")
        print("Model successfully compiled for optimised performance.")
        return compiled
    except Exception as e:
        print(f"torch.compile failed: {e}. Returning standard model.")
        return model

In [ ]:
model = make_fast_predictor().to(device)

Check the model

In [ ]:
model(torch.zeros([1,10]).to(device))

### Benchmarking

Benchmark the evaluation speed of the model

In [30]:
NUM_BENCHMARKS = 10000

import torch
import time
model.eval()
x = torch.rand((1, 10), device=device)
results = []

with torch.no_grad():

    # warmup
    for _ in range(1000):
        model(x)

    for _ in range (NUM_BENCHMARKS):
        torch.cuda.synchronize()
        start = time.perf_counter()
        model(x)
        end = time.perf_counter()
        results.append((end-start)*1000)

per_pred_us = sum(results) / NUM_BENCHMARKS * 1e3
print(f"Per prediction: {per_pred_us:.2f} ms")

Per prediction: 773.60 ms


In [31]:
print(f"\n=== Benchmark Results ({device}) ===")
print(f"Max: {max(results)} ms")
print(f"Min: {min(results)} ms")
print(f"Avg: {sum(results) / len(results)} ms" )
print(f"Median: {sorted(results)[len(results) // 2]} ms")
print("Results:")
print(results)


=== Benchmark Results (cuda) ===
Max: 1.5046359999359993 ms
Min: 0.6963010000617942 ms
Avg: 0.773595204001731 ms
Median: 0.7518319998780498 ms
Results:
[0.7548140001745196, 0.7981260000633483, 0.8547990000806749, 0.7957510001688206, 0.8098419998532336, 0.7987609997144318, 0.7384099999399041, 0.7793479999236297, 0.793906999660976, 0.8129869997901551, 0.7798289998390828, 0.740468999993027, 0.7310080000024755, 0.771573999827524, 0.8433960001639207, 0.8027889998629689, 0.7567909997305833, 0.7208000001810433, 0.7559320001746528, 0.766214000123, 0.7596549999107083, 0.7430710002154228, 0.732567999875755, 0.7567870002276322, 0.7593589998577954, 0.757173999772931, 0.8004829996934859, 0.8402109997405205, 0.7585029998153914, 0.8109779996630095, 0.7628009998370544, 0.7554459998573293, 0.875229000030231, 0.742554999760614, 0.7728850000603416, 0.7542209996245219, 0.7598600000164879, 0.7321130001400888, 0.7473820000996056, 0.8360730003005301, 0.8267680000244582, 0.7865779998610378, 0.729795000097510

## 4. Loss

Huber (smooth-L1) loss replaces L1.  Near zero it's quadratic (smooth gradients,
won't over-punish tiny residuals); far from zero it's linear (robust to the
occasional outlier).  `delta=1.0` is in *normalized* target units so it's
roughly one standard deviation of the target.

**Energy-conservation penalty.**  For each collision sample we integrate one
timestep forward using the *predicted* wrench and check whether the body's
kinetic energy would grow beyond `e^2 * KE_before` (where `e` is the
coefficient of restitution).  Any excess is squared and added to the loss,
so the term is zero for physically admissible predictions and grows smoothly
when the model would inject energy — the exact failure mode you were seeing
at low collision speeds.  You control it with `w_energy`, `dt`, `mass`,
`inertia_diag`, and `restitution` when constructing `WrenchLoss`.


In [32]:
class WrenchLoss(nn.Module):
    """Physics-structured loss for a Hooke (linear elastic) contact model.

    Components:
        - BCE on collision_logit (with pos_weight for class imbalance).
        - Masked Huber on force and torque, operating in PHYSICAL units.
        - Soft penalty on f_n < 0 (Signorini violation).
        - Soft penalty on energy gain during collision (restitution-aware),
          using a BOUNDED log-based term so large violations at init don't
          produce enormous gradients that kill regression learning.

    Energy-conservation term
    ------------------------
    Given a predicted force F and torque tau applied over one timestep dt to a
    body with mass m and (diagonal) inertia I, the post-step velocities are
        v'   = v   + (F   / m) * dt
        w'   = w   + (I^-1 tau) * dt
    and the kinetic energy is  KE = 0.5 m |v|^2 + 0.5 w^T I w.
    For a real collision with coefficient of restitution e in [0, 1] we expect
        KE'  <=  e^2 * KE_before.
    We define the energy ratio
        r = KE' / (e^2 * KE_before)
    and penalise log(max(r, 1))^2. This is zero when r <= 1 (admissible),
    grows like (log r)^2 when r > 1, and its gradient in r is bounded — so a
    random-init network that predicts wildly wrong forces at epoch 0 won't
    produce an exploding energy gradient that pushes the model into the
    degenerate F≈0 basin.

    Warmup: w_energy typically starts at 0 and is ramped up over several
    epochs by the training loop via `set_energy_weight`, so the regression
    heads (force, torque) get to learn first before conservation pressure
    kicks in.

    Because targets are NOT pre-normalised, you may need to set w_force and
    w_torque to bring the two regression terms to comparable magnitude. A good
    heuristic is to set:
        w_force  ~ 1 / (force_rms_in_physical_units)
        w_torque ~ 1 / (torque_rms_in_physical_units)
    so both contribute roughly equally early in training.
    """
    def __init__(self,
                 w_force=1.0, w_torque=1.0, w_collision=1.0,
                 w_energy=0.0,
                 huber_delta=1.0, pos_weight=None,
                 dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                 restitution=0.5):
        super().__init__()
        self.w_force     = w_force
        self.w_torque    = w_torque
        self.w_collision = w_collision
        self.w_energy    = w_energy
        self.huber_delta = huber_delta
        self.dt          = float(dt)
        self.mass        = float(mass)
        self.restitution = float(restitution)
        # Inertia tensor (diagonal) for a unit cube by default: I = (1/6) m a^2
        # with m=1, a=1. Override via the constructor to match your simulated body.
        self.register_buffer(
            "inertia_diag",
            torch.tensor(inertia_diag, dtype=torch.float32),
        )
        # pos_weight is a tensor; register as buffer so .to(device) moves it.
        if pos_weight is not None and not torch.is_tensor(pos_weight):
            pos_weight = torch.tensor(float(pos_weight))
        self.register_buffer(
            "pos_weight",
            pos_weight if pos_weight is not None else torch.tensor(1.0),
        )
        self._has_pos_weight = pos_weight is not None

    def set_energy_weight(self, w):
        """Runtime hook for the training loop's warmup schedule."""
        self.w_energy = float(w)

    def _kinetic_energy(self, v, w):
        """KE = 0.5 m |v|^2 + 0.5 w^T I w for diagonal I. Shapes: (B,3)."""
        ke_lin = 0.5 * self.mass * (v * v).sum(dim=-1, keepdim=True)
        ke_rot = 0.5 * (self.inertia_diag * w * w).sum(dim=-1, keepdim=True)
        return ke_lin + ke_rot

    def forward(self, preds, targets):
        mask   = targets["is_collision"]                   # (B, 1)
        n_coll = mask.sum().clamp_min(1.0)

        # --- collision BCE ---
        loss_collision = F.binary_cross_entropy_with_logits(
            preds["collision_logit"], mask.float(),
            pos_weight=self.pos_weight if self._has_pos_weight else None,
            reduction="mean",
        )
        # --- masked Huber on force and torque (physical units) ---
        raw_f = F.huber_loss(preds["force"],  targets["force"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        raw_t = F.huber_loss(preds["torque"], targets["torque"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        loss_force  = (raw_f.sum(dim=-1, keepdim=True) * mask).sum() / n_coll
        loss_torque = (raw_t.sum(dim=-1, keepdim=True) * mask).sum() / n_coll

        # --- energy-conservation penalty (bounded, log-based) ---
        # Integrate one step using the predicted wrench and compare KE before vs after.
        # Only collision samples contribute (non-contact steps have F=tau=0 anyway).
        v = targets["lin_vel"]                              # (B, 3)
        w = targets["ang_vel"]                              # (B, 3)
        f_pred = preds["force"]                             # (B, 3)
        t_pred = preds["torque"]                            # (B, 3)

        v_next = v + (f_pred / self.mass) * self.dt
        # Diagonal inertia -> element-wise divide
        w_next = w + (t_pred / self.inertia_diag) * self.dt

        ke_before = self._kinetic_energy(v, w)              # (B, 1)
        ke_after  = self._kinetic_energy(v_next, w_next)    # (B, 1)

        # Log-ratio penalty:
        #   r = ke_after / (e^2 * ke_before + eps),  penalty = max(log r, 0)^2
        # Bounded gradient in F: d/dF log(ke_after) scales as 1/ke_after, so at
        # init where ke_after is huge the gradient is SMALL — the opposite of
        # (ke_after - budget)^2, which has gradient proportional to ke_after.
        eps       = 1e-6
        ke_budget = (self.restitution ** 2) * ke_before
        log_ratio = torch.log(ke_after + eps) - torch.log(ke_budget + eps)
        excess_log  = F.relu(log_ratio)                     # zero when admissible
        loss_energy = ((excess_log ** 2) * mask).sum() / n_coll

        total = (self.w_force     * loss_force
               + self.w_torque    * loss_torque
               + self.w_collision * loss_collision
               + self.w_energy    * loss_energy)

        # Diagnostic: fraction of collision samples that currently violate conservation,
        # plus the geometric-mean ratio so you can see HOW MUCH they violate by.
        with torch.no_grad():
            violating = ((excess_log > 0).float() * mask).sum() / n_coll
            # Mean log-ratio over collision samples (in log space so it's well-behaved)
            mean_log_ratio = (log_ratio * mask).sum() / n_coll

        return total, {
            "force":             loss_force.item(),
            "torque":            loss_torque.item(),
            "collision":         loss_collision.item(),
            "energy":            loss_energy.item(),
            "energy_violating":  violating.item(),
            "energy_mean_logr":  mean_log_ratio.item(),
            "w_energy":          self.w_energy,
            "total":             total.item(),
            "active_collisions": n_coll.item(),
            "k":                 preds["aux"]["k"].item(),
        }

## Training

In [33]:
def train_model(model, train_loader, val_loader, train_dataset,
                epochs=200, lr=1e-4, weight_decay=1e-4,
                w_force=1.0, w_torque=1.0, w_collision=0.5,
                # Energy-conservation warmup schedule.
                # Rationale: starting with w_energy>0 produces enormous gradients
                # at init (ke_after is huge for a random-init network) and pushes
                # the model into the degenerate F≈0 basin, where regression loss
                # plateaus at force_rms. Warming up from 0 lets the force/torque
                # heads learn a reasonable solution first; only then do we
                # gently tighten conservation.
                w_energy_max=0.1, energy_warmup_start=20, energy_warmup_epochs=30,
                dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                restitution=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Class-imbalance weight for BCE: #no-contact / #contact on train set.
    collisions = train_dataset.collisions.squeeze(-1).bool()
    n_pos = int(collisions.sum().item())
    n_neg = int((~collisions).sum().item())
    pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0
    print(f"BCE pos_weight = {pos_weight:.3f}  ({n_pos} contacts / {n_neg} non-contacts)")

    criterion = WrenchLoss(
        w_force=w_force, w_torque=w_torque, w_collision=w_collision,
        w_energy=0.0,  # ramped up by the warmup schedule below
        dt=dt, mass=mass, inertia_diag=inertia_diag, restitution=restitution,
        huber_delta=1.0, pos_weight=pos_weight,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    def energy_weight_at(epoch):
        """Linear ramp from 0 to w_energy_max over [start, start+epochs)."""
        if epoch < energy_warmup_start:
            return 0.0
        if energy_warmup_epochs <= 0:
            return float(w_energy_max)
        frac = (epoch - energy_warmup_start) / float(energy_warmup_epochs)
        return float(w_energy_max) * min(max(frac, 0.0), 1.0)

    best_val_loss = float('inf')
    loss_keys = ['total', 'force', 'torque', 'collision',
                 'energy', 'energy_violating', 'energy_mean_logr']

    for epoch in tqdm(range(epochs)):
        criterion.set_energy_weight(energy_weight_at(epoch))

        # --- Train ---
        model.train()
        train_losses = {k: 0.0 for k in loss_keys}
        for features, targets in train_loader:
            features = features.to(device)
            out = model(features)
            targets  = {k: v.to(device) for k, v in targets.items()}
            optimizer.zero_grad()

            loss, components = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            for k in loss_keys:
                train_losses[k] += components[k]
        for k in loss_keys:
            train_losses[k] /= len(train_loader)

        # --- Validate ---
        model.eval()
        val_losses = {k: 0.0 for k in loss_keys}
        with torch.no_grad():
            for features, targets in val_loader:
                features = features.to(device)
                targets  = {k: v.to(device) for k, v in targets.items()}
                _, components = criterion(model(features), targets)
                for k in loss_keys:
                    val_losses[k] += components[k]
        for k in loss_keys:
            val_losses[k] /= len(val_loader)

        scheduler.step(val_losses['total'])

        if val_losses['total'] < best_val_loss:
            best_val_loss = val_losses['total']
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss':        best_val_loss,
            }, 'wrench_model_best.pth')

        if (epoch + 1) % 10 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            k_now  = components.get('k', float('nan'))
            we_now = criterion.w_energy
            print(f"Epoch {epoch+1}/{epochs} | LR: {lr_now:.2e} | k={k_now:.1f} | w_energy={we_now:.4f}")
            print(f"  Train: total={train_losses['total']:.4f} "
                  f"f={train_losses['force']:.4f} t={train_losses['torque']:.4f} "
                  f"c={train_losses['collision']:.4f} "
                  f"e={train_losses['energy']:.4f} vio={train_losses['energy_violating']:.2f} "
                  f"logr={train_losses['energy_mean_logr']:+.3f}")
            print(f"  Val:   total={val_losses['total']:.4f} "
                  f"f={val_losses['force']:.4f} t={val_losses['torque']:.4f} "
                  f"c={val_losses['collision']:.4f} "
                  f"e={val_losses['energy']:.4f} vio={val_losses['energy_violating']:.2f} "
                  f"logr={val_losses['energy_mean_logr']:+.3f}")

    return model

### Execute training

In [34]:
model = train_model(model, train_loader, val_loader, full_dataset,
                    epochs=200, lr=1e-3,
                    w_force=1.0, w_torque=1.0, w_collision=0.5)


BCE pos_weight = 0.723  (969053 contacts / 700869 non-contacts)


  5%|▌         | 10/200 [03:38<1:08:05, 21.50s/it]

Epoch 10/200 | LR: 1.00e-03 | k=1000.0 | w_energy=0.0000
  Train: total=533.0724 f=325.5701 t=207.3885 c=0.2276 e=46.1524 vio=1.00 logr=+5.814
  Val:   total=508.2468 f=307.7247 t=200.4217 c=0.2008 e=46.2860 vio=1.00 logr=+5.841


 10%|█         | 20/200 [07:09<1:03:23, 21.13s/it]

Epoch 20/200 | LR: 1.00e-03 | k=1000.0 | w_energy=0.0000
  Train: total=286.3074 f=102.8879 t=183.3275 c=0.1839 e=48.7358 vio=1.00 logr=+5.979
  Val:   total=260.1149 f=80.3833 t=179.6454 c=0.1725 e=48.1046 vio=1.00 logr=+5.933


...

Epoch 180/200 | LR: 6.25e-05 | k=1000.0 | w_energy=0.1000
  Train: total=24.0113 f=7.1072 t=11.9070 c=0.0316 e=49.8128 vio=1.00 logr=+6.064
  Val:   total=31.1038 f=12.5961 t=13.5293 c=0.0312 e=49.6277 vio=1.00 logr=+6.053


 95%|█████████▌| 190/200 [1:06:07<03:30, 21.03s/it]

Epoch 190/200 | LR: 6.25e-05 | k=1000.0 | w_energy=0.1000
  Train: total=24.5307 f=7.6788 t=11.8553 c=0.0314 e=49.8099 vio=1.00 logr=+6.064
  Val:   total=31.1044 f=12.6395 t=13.4871 c=0.0315 e=49.6205 vio=1.00 logr=+6.054


100%|██████████| 200/200 [1:09:38<00:00, 20.89s/it]

Epoch 200/200 | LR: 3.13e-05 | k=1000.0 | w_energy=0.1000
  Train: total=23.0719 f=6.5984 t=11.4768 c=0.0303 e=49.8157 vio=1.00 logr=+6.065
  Val:   total=30.2066 f=11.9722 t=13.2578 c=0.0304 e=49.6136 vio=1.00 logr=+6.050


## Evaluation

In [35]:
@torch.no_grad()
def evaluate(model, loader, dataset, device="cuda",
             rel_floor_force=0.05, rel_floor_torque=0.05):
    """Evaluate in physical units (targets were not normalised).

    Args:
        dataset: the *underlying* ContactDataset (not a Subset). Kept for API
                 symmetry; no target stats are needed since targets are physical.
        rel_floor_{force,torque}: targets with magnitude below this (in
                 physical units) are excluded from the relative-error stats
                 to avoid division-by-near-zero blow-up.
    """
    model.eval()
    all_abs_f, all_abs_t = [], []
    all_rel_f, all_rel_t = [], []
    all_coll_correct     = []

    # Physics-diagnostic accumulators (frictionless model: only f_n sign check)
    n_fn_neg = 0
    n_total  = 0

    for features, targets in loader:
        features = features.to(device)
        f_tgt = targets["force"].to(device)
        t_tgt = targets["torque"].to(device)
        c_tgt = targets["is_collision"].to(device).squeeze(-1).bool()

        preds = model(features)
        f_pred = preds["force"]
        t_pred = preds["torque"]
        c_pred = (torch.sigmoid(preds["collision_logit"]).squeeze(-1) > 0.5)

        all_coll_correct.append((c_pred == c_tgt).float().cpu())

        if c_tgt.any():
            f_pred_c = f_pred[c_tgt]
            f_tgt_c  = f_tgt[c_tgt]
            t_pred_c = t_pred[c_tgt]
            t_tgt_c  = t_tgt[c_tgt]

            # Targets are already physical — no de-normalisation needed.
            abs_f = (f_pred_c - f_tgt_c).norm(dim=-1)
            abs_t = (t_pred_c - t_tgt_c).norm(dim=-1)
            all_abs_f.append(abs_f.cpu())
            all_abs_t.append(abs_t.cpu())

            f_norm = f_tgt_c.norm(dim=-1)
            t_norm = t_tgt_c.norm(dim=-1)
            mask_f = f_norm > rel_floor_force
            mask_t = t_norm > rel_floor_torque
            if mask_f.any():
                rel_f = (f_pred_c[mask_f] - f_tgt_c[mask_f]).norm(dim=-1) / f_norm[mask_f]
                all_rel_f.append(rel_f.cpu())
            if mask_t.any():
                rel_t = (t_pred_c[mask_t] - t_tgt_c[mask_t]).norm(dim=-1) / t_norm[mask_t]
                all_rel_t.append(rel_t.cpu())

            # Physics diagnostic: how often the Hooke-plus-residual allows f_n < 0.
            f_n_c = preds["aux"]["force_residual"][c_tgt]
            n_fn_neg += int((f_n_c < 0).sum().item())
            n_total  += int(c_tgt.sum().item())

    abs_f = torch.cat(all_abs_f) if all_abs_f else torch.empty(0)
    abs_t = torch.cat(all_abs_t) if all_abs_t else torch.empty(0)
    rel_f = torch.cat(all_rel_f) if all_rel_f else torch.empty(0)
    rel_t = torch.cat(all_rel_t) if all_rel_t else torch.empty(0)
    coll_acc = torch.cat(all_coll_correct).mean().item()

    def fmt_pct(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.1%}  p90={e.quantile(0.9):.1%}  "
                f"p99={e.quantile(0.99):.1%}")
    def fmt_abs(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.4f}  p90={e.quantile(0.9):.4f}  "
                f"p99={e.quantile(0.99):.4f}")

    print("─" * 60)
    print(f"Collision accuracy : {coll_acc:.3%}")
    print(f"Force  abs err     : {fmt_abs(abs_f)}")
    print(f"Torque abs err     : {fmt_abs(abs_t)}")
    print(f"Force  rel err     : {fmt_pct(rel_f)}  "
          f"(on {rel_f.numel()} / {abs_f.numel()} samples above floor)")
    print(f"Torque rel err     : {fmt_pct(rel_t)}  "
          f"(on {rel_t.numel()} / {abs_t.numel()} samples above floor)")
    if n_total > 0:
        print(f"Physics violations : f_n<0 in {n_fn_neg}/{n_total} "
              f"({100*n_fn_neg/n_total:.2f}%)")
    print("─" * 60)

    return {
        "collision_acc":  coll_acc,
        "force_abs_p50":  abs_f.median().item() if abs_f.numel() else float("nan"),
        "force_abs_p99":  abs_f.quantile(0.99).item() if abs_f.numel() else float("nan"),
        "torque_abs_p50": abs_t.median().item() if abs_t.numel() else float("nan"),
        "torque_abs_p99": abs_t.quantile(0.99).item() if abs_t.numel() else float("nan"),
        "fn_neg_rate":    (n_fn_neg / n_total) if n_total > 0 else float("nan"),
    }

## Print 100 data points

In [36]:
@torch.no_grad()
def print_predictions(model, loader, n=100, device="cuda"):
    """Print n predictions vs ground truth from the loader."""
    model.eval()

    all_f_pred, all_f_tgt = [], []
    all_t_pred, all_t_tgt = [], []
    all_coll_pred, all_coll_tgt = [], []
    all_D, all_power = [], []

    for features, targets in loader:
        features = features.to(device)
        preds = model(features)

        all_f_pred.append(preds["force"].cpu())
        all_t_pred.append(preds["torque"].cpu())
        all_f_tgt.append(targets["force"])
        all_t_tgt.append(targets["torque"])
        all_coll_pred.append(torch.sigmoid(preds["collision_logit"]).cpu())
        all_coll_tgt.append(targets["is_collision"])

        collected = sum(x.shape[0] for x in all_f_pred)
        if collected >= n:
            break

    f_pred = torch.cat(all_f_pred)[:n]
    f_tgt  = torch.cat(all_f_tgt)[:n]
    t_pred = torch.cat(all_t_pred)[:n]
    t_tgt  = torch.cat(all_t_tgt)[:n]
    c_pred = torch.cat(all_coll_pred)[:n].squeeze(-1)
    c_tgt  = torch.cat(all_coll_tgt)[:n].squeeze(-1)

    header = (f"{'#':>4s}  {'coll':>5s} {'pred':>5s}  "
              f"{'force_pred':>30s}  {'force_true':>30s}  "
              f"{'torque_pred':>30s}  {'torque_true':>30s}  ")
    print(header)
    print("─" * len(header))

    for i in range(n):
        cp = f"{c_pred[i]:.2f}"
        ct = f"{int(c_tgt[i].item())}"
        fp = f"[{f_pred[i,0]:8.3f}, {f_pred[i,1]:8.3f}, {f_pred[i,2]:8.3f}]"
        ft = f"[{f_tgt[i,0]:8.3f}, {f_tgt[i,1]:8.3f}, {f_tgt[i,2]:8.3f}]"
        tp = f"[{t_pred[i,0]:8.4f}, {t_pred[i,1]:8.4f}, {t_pred[i,2]:8.4f}]"
        tt = f"[{t_tgt[i,0]:8.4f}, {t_tgt[i,1]:8.4f}, {t_tgt[i,2]:8.4f}]"
        print(f"{i:4d}  {ct:>5s} {cp:>5s}  {fp:>30s}  {ft:>30s}  {tp:>30s}  {tt:>30s}")

print_predictions(model, val_loader, n=100)


   #   coll  pred                      force_pred                      force_true                     torque_pred                     torque_true  
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   0      0  0.00  [   0.423,    0.140, -5945.037]  [   0.000,    0.000,    0.000]  [366.6675,  93.4022,   0.0283]  [  0.0000,   0.0000,   0.0000]
   1      0  0.00  [   0.002,    0.000,    0.707]  [   0.000,    0.000,    0.000]  [  0.1701,   0.0659,  -0.0004]  [  0.0000,   0.0000,   0.0000]
   2      0  0.00  [   0.301,    0.419, -5057.519]  [   0.000,    0.000,    0.000]  [350.3177, 479.7009,   0.0606]  [  0.0000,   0.0000,   0.0000]
   3      1  1.00  [  -0.105,   -0.061, -13595.451]  [   0.000,    0.000, -13579.935]  [-837.0577, 1620.6896,  -0.0009]  [-788.7222, 1725.2313,   0.0000]
   4      1  0.99  [   0.012,   -0.001,  158.329]  [   0.000,    0.000,  160.793]  [  1.7320, -26.7677,  -0.00

Download checkpoint (Colab)

In [37]:
if is_colab():
    from google.colab import files
    files.download("wrench_model_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>